# Notebook 07 — Visualization: Enhanced 3D ResUNet RibFrac

This notebook visualizes outputs from Notebook 05/06 for the current RibFrac128 development smoke test.

It produces:
- training/validation loss curves;
- classification accuracy curves;
- segmentation Dice curves when available;
- classification confusion matrix;
- classification probability plots;
- CT, ground-truth, and predicted segmentation figures;
- predicted segmentation overlays;
- a visualization summary.

**Important:** Current RibFrac128 results are development/smoke-test results, not final thesis results.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")


In [ ]:
TRAINING_RESULTS_ROOT = Path("/kaggle/input/datasets/waboke/ribfrac-training-results")
PATCH_DATASET_ROOT = Path("/kaggle/input/datasets/waboke/ribfrac-patch-dataset")
EVALUATION_ROOT = Path("/kaggle/working/evaluation_results")
VISUALIZATION_ROOT = Path("/kaggle/working/visualization_results")
VISUALIZATION_ROOT.mkdir(parents=True, exist_ok=True)

print("Training results:", TRAINING_RESULTS_ROOT)
print("Patch dataset:", PATCH_DATASET_ROOT)
print("Evaluation results:", EVALUATION_ROOT)
print("Visualization output:", VISUALIZATION_ROOT)


In [ ]:
required_files = [
    "training_history.csv",
    "experiment_config.json",
    "final_enhanced_3d_resunet_multitask.keras",
]
for name in required_files:
    path = TRAINING_RESULTS_ROOT / name
    print(f"{name}: {path.exists()}")

print("Evaluation directory exists:", EVALUATION_ROOT.exists())


In [ ]:
CONFIG_PATH = TRAINING_RESULTS_ROOT / "experiment_config.json"
with open(CONFIG_PATH, "r") as f:
    experiment_config = json.load(f)

print(json.dumps(experiment_config, indent=4))


## 1. Training and Validation Loss

In [ ]:
HISTORY_CSV = TRAINING_RESULTS_ROOT / "training_history.csv"
history_df = pd.read_csv(HISTORY_CSV) if HISTORY_CSV.exists() else pd.DataFrame()

print("Training history shape:", history_df.shape)
print(history_df.columns.tolist())

if not history_df.empty:
    display(history_df.head())


In [ ]:
if not history_df.empty and "loss" in history_df.columns:
    plt.figure(figsize=(8,5))
    plt.plot(history_df.index + 1, history_df["loss"], label="Training Loss")
    if "val_loss" in history_df.columns:
        plt.plot(history_df.index + 1, history_df["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = VISUALIZATION_ROOT / "training_validation_loss.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)
else:
    print("Loss columns are not available.")


## 2. Classification Accuracy

In [ ]:
train_acc_candidates = [
    "classification_output_accuracy",
    "classification_accuracy",
    "classification_output_sparse_categorical_accuracy",
]
val_acc_candidates = [
    "val_classification_output_accuracy",
    "val_classification_accuracy",
    "val_classification_output_sparse_categorical_accuracy",
]

train_acc = next((c for c in train_acc_candidates if c in history_df.columns), None)
val_acc = next((c for c in val_acc_candidates if c in history_df.columns), None)

print("Training accuracy column:", train_acc)
print("Validation accuracy column:", val_acc)

if train_acc or val_acc:
    plt.figure(figsize=(8,5))
    if train_acc:
        plt.plot(history_df.index + 1, history_df[train_acc], label="Training Accuracy")
    if val_acc:
        plt.plot(history_df.index + 1, history_df[val_acc], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Classification Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = VISUALIZATION_ROOT / "classification_accuracy.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)
else:
    print("Classification accuracy was not recorded.")


## 3. Segmentation Dice

In [ ]:
dice_candidates = [
    "segmentation_output_dice_coefficient",
    "segmentation_output_dice",
    "dice_coefficient",
    "dice",
]
val_dice_candidates = [
    "val_segmentation_output_dice_coefficient",
    "val_segmentation_output_dice",
    "val_dice_coefficient",
    "val_dice",
]

train_dice = next((c for c in dice_candidates if c in history_df.columns), None)
val_dice = next((c for c in val_dice_candidates if c in history_df.columns), None)

print("Training Dice column:", train_dice)
print("Validation Dice column:", val_dice)

if train_dice or val_dice:
    plt.figure(figsize=(8,5))
    if train_dice:
        plt.plot(history_df.index + 1, history_df[train_dice], label="Training Dice")
    if val_dice:
        plt.plot(history_df.index + 1, history_df[val_dice], label="Validation Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title("Segmentation Dice")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = VISUALIZATION_ROOT / "segmentation_dice.png"
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)
else:
    print("Dice was not recorded in training history.")


## 4. Load Development Validation Data and Model

In [ ]:
import tensorflow as tf

RIBFRAC128_ROOT = PATCH_DATASET_ROOT / "RibFrac128"
X_val = np.load(RIBFRAC128_ROOT / "X.npy")
Y_seg_val = np.load(RIBFRAC128_ROOT / "Y_seg.npy")
Y_class_val = np.load(RIBFRAC128_ROOT / "Y_class.npy")

MODEL_PATH = TRAINING_RESULTS_ROOT / "final_enhanced_3d_resunet_multitask.keras"
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print("X:", X_val.shape)
print("Y_seg:", Y_seg_val.shape)
print("Y_class:", Y_class_val.shape)
print("Model input:", model.input_shape)
print("Model outputs:", model.output_shape)


In [ ]:
predictions = model.predict(X_val, batch_size=1, verbose=1)

classification_probabilities = predictions[0]
segmentation_probabilities = predictions[1]

y_true_class = np.asarray(Y_class_val).astype(int).reshape(-1)
y_pred_class = np.argmax(classification_probabilities, axis=1)

y_true_seg = (np.asarray(Y_seg_val) >= 0.5).astype(np.uint8)
y_pred_seg = (segmentation_probabilities >= 0.5).astype(np.uint8)

class_names = [
    "Background",
    "Displaced",
    "Non-displaced",
    "Buckle",
    "Segmental",
]

print("Classification predictions:", classification_probabilities.shape)
print("Segmentation predictions:", segmentation_probabilities.shape)
print("True classes:", y_true_class)
print("Predicted classes:", y_pred_class)


## 5. Classification Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(
    y_true_class,
    y_pred_class,
    labels=np.arange(len(class_names))
)

print(cm)

fig, ax = plt.subplots(figsize=(8,7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("Classification Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

path = VISUALIZATION_ROOT / "classification_confusion_matrix.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 6. Classification Prediction Probabilities

In [ ]:
n_samples = classification_probabilities.shape[0]
x = np.arange(len(class_names))

fig, ax = plt.subplots(figsize=(10,5))
width = 0.8 / max(n_samples, 1)

for i in range(n_samples):
    offset = (i - (n_samples - 1)/2) * width
    ax.bar(
        x + offset,
        classification_probabilities[i],
        width=width,
        label=f"Patch {i+1}"
    )

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.set_ylabel("Predicted Probability")
ax.set_ylim(0,1)
ax.set_title("Classification Prediction Probabilities")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

path = VISUALIZATION_ROOT / "classification_probabilities.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 7. Representative Segmentation Visualization

In [ ]:
sample_index = 0

sample_ct = X_val[sample_index, ..., 0]
sample_true = y_true_seg[sample_index, ..., 0]
sample_pred = y_pred_seg[sample_index, ..., 0]

true_voxels_per_slice = sample_true.sum(axis=(0,1))

if true_voxels_per_slice.max() > 0:
    slice_index = int(np.argmax(true_voxels_per_slice))
else:
    pred_voxels_per_slice = sample_pred.sum(axis=(0,1))
    slice_index = (
        int(np.argmax(pred_voxels_per_slice))
        if pred_voxels_per_slice.max() > 0
        else sample_ct.shape[2] // 2
    )

ct_slice = sample_ct[:, :, slice_index]
true_slice = sample_true[:, :, slice_index]
pred_slice = sample_pred[:, :, slice_index]

print("Patch:", sample_index)
print("Axial slice:", slice_index)
print("Ground-truth voxels:", int(true_slice.sum()))
print("Predicted voxels:", int(pred_slice.sum()))


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))

axes[0].imshow(ct_slice.T, cmap="gray", origin="lower")
axes[0].set_title("CT Slice")
axes[0].axis("off")

axes[1].imshow(ct_slice.T, cmap="gray", origin="lower")
axes[1].imshow(true_slice.T, alpha=0.45, origin="lower")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

axes[2].imshow(ct_slice.T, cmap="gray", origin="lower")
axes[2].imshow(pred_slice.T, alpha=0.45, origin="lower")
axes[2].set_title("Predicted Segmentation")
axes[2].axis("off")

plt.suptitle(f"Representative RibFrac128 Patch — Axial Slice {slice_index}")
plt.tight_layout()

path = VISUALIZATION_ROOT / "segmentation_comparison.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 8. Predicted Segmentation Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(7,7))

ax.imshow(ct_slice.T, cmap="gray", origin="lower")
ax.imshow(
    np.ma.masked_where(pred_slice.T == 0, pred_slice.T),
    alpha=0.5,
    origin="lower"
)

ax.set_title(f"Predicted Fracture Overlay — Slice {slice_index}")
ax.axis("off")
plt.tight_layout()

path = VISUALIZATION_ROOT / "predicted_segmentation_overlay.png"
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 9. Visualization Summary

In [ ]:
summary = {
    "validation_patches": int(len(X_val)),
    "representative_patch": int(sample_index),
    "axial_slice": int(slice_index),
    "ground_truth_voxels_3d": int(sample_true.sum()),
    "predicted_voxels_3d": int(sample_pred.sum()),
    "ground_truth_voxels_slice": int(true_slice.sum()),
    "predicted_voxels_slice": int(pred_slice.sum()),
    "true_class": class_names[int(y_true_class[sample_index])],
    "predicted_class": class_names[int(y_pred_class[sample_index])],
    "predicted_class_probability": float(
        classification_probabilities[
            sample_index,
            y_pred_class[sample_index]
        ]
    ),
}

with open(VISUALIZATION_ROOT / "visualization_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))


## 10. Final Output Inventory

In [ ]:
files = sorted(
    p.relative_to(VISUALIZATION_ROOT)
    for p in VISUALIZATION_ROOT.rglob("*")
    if p.is_file()
)

print("=" * 60)
print("NOTEBOOK 07 — VISUALIZATION COMPLETED")
print("=" * 60)

for f in files:
    print(f)

print("\nSaved to:", VISUALIZATION_ROOT)
print("\nThese are development/smoke-test visualizations, not final thesis figures.")
print("=" * 60)
